In [1]:
import os
import re
import shutil
import random
import subprocess
import sys

In [2]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Total-Data-20260603T163352Z-3-001.zip"

ROOT          = "/content/carixray"
ZIP_PATH      = os.path.join(ROOT, "Total-Data.zip")
EXTRACT_DIR   = os.path.join(ROOT, "raw_data")
DET_KFOLD_DIR = os.path.join(ROOT, "DETECTION/K_folds")
SEG_KFOLD_DIR = os.path.join(ROOT, "SEGMENTATION/K_folds")
# Set them to None to let the script attempt auto-detection.
DET_TOTAL_DIR_OVERRIDE = None  # e.g., "/content/carixray/raw_data/Total-Data/DETECTION/DETECTION"
SEG_TOTAL_DIR_OVERRIDE = None  # e.g., "/content/carixray/raw_data/Total-Data/SEGMENTATION/SEGMENTATION"

N_FOLDS    = 5
SEED       = 42          # added for reproducibility (original script is unseeded)
EPOCHS_DET = 15
EPOCHS_SEG = 15
IMG_SIZE   = 640
DET_MODEL  = "yolov8n.pt"
SEG_MODEL  = "yolov8n-seg.pt"
CLASS_NAMES = ["caries"]


In [3]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip_install("ultralytics", "pyyaml", "pandas")

import yaml
import pandas as pd


In [4]:
# 1. Mount Google Drive + copy/extract dataset
# ----------------------------------------------------------------------------
os.makedirs(ROOT, exist_ok=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab — skipping drive.mount(). "
          "Make sure DRIVE_ZIP_PATH is accessible locally.")

if not os.path.exists(DRIVE_ZIP_PATH):
    raise FileNotFoundError(
        f"Could not find dataset zip at: {DRIVE_ZIP_PATH}\n"
        "Set DRIVE_ZIP_PATH to the correct path inside your mounted Drive "
        "(browse the file panel on the left in Colab, right-click the file > 'Copy path')."
    )

if not os.path.exists(ZIP_PATH) or (os.path.exists(DRIVE_ZIP_PATH) and os.path.getsize(ZIP_PATH) != os.path.getsize(DRIVE_ZIP_PATH)):
    print(f"Copying dataset zip from Drive ({DRIVE_ZIP_PATH}) to local storage...")
    if os.path.exists(ZIP_PATH):
        try: os.remove(ZIP_PATH)
        except: pass
    shutil.copy(DRIVE_ZIP_PATH, ZIP_PATH)

if not os.path.exists(EXTRACT_DIR) or len(os.listdir(EXTRACT_DIR)) == 0:
    print("Extracting dataset...")
    if os.path.exists(EXTRACT_DIR):
        shutil.rmtree(EXTRACT_DIR)
    shutil.unpack_archive(ZIP_PATH, EXTRACT_DIR)

# Automatically extract any nested zip files
def extract_nested_archives(root):
    extracted_any = False
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            if f.lower().endswith(".zip"):
                zip_file = os.path.join(dirpath, f)
                # Extract to a folder named after the zip file
                target_dir = os.path.join(dirpath, os.path.splitext(f)[0])
                print(f"Found nested zip: {zip_file}. Extracting to {target_dir}...")
                try:
                    shutil.unpack_archive(zip_file, target_dir)
                    os.remove(zip_file)
                    extracted_any = True
                except Exception as e:
                    print(f"Failed to extract {zip_file}: {e}")
    if extracted_any:
        extract_nested_archives(root)

extract_nested_archives(EXTRACT_DIR)

print("Extraction complete. Top-level contents:")
for item in os.listdir(EXTRACT_DIR):
    print("  -", item)

Mounted at /content/drive
Copying dataset zip from Drive (/content/drive/MyDrive/Total-Data-20260603T163352Z-3-001.zip) to local storage...
Extracting dataset...
Found nested zip: /content/carixray/raw_data/Total-Data/DETECTION.zip. Extracting to /content/carixray/raw_data/Total-Data/DETECTION...
Found nested zip: /content/carixray/raw_data/Total-Data/SEGMENTATION.zip. Extracting to /content/carixray/raw_data/Total-Data/SEGMENTATION...
Extraction complete. Top-level contents:
  - Total-Data


In [5]:
# ----------------------------------------------------------------------------
# 2. Auto-discover DETECTION / SEGMENTATION folders
#    (any folder containing an "images" subdir and a "labels" subdir)
# ----------------------------------------------------------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def find_candidate_dirs(root):
    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        # Normalize folder names to lowercase to be case-insensitive
        lower_dirnames = [d.lower() for d in dirnames]
        if "images" in lower_dirnames and "labels" in lower_dirnames:
            candidates.append(dirpath)
    return candidates

def detect_label_type(labels_dir, sample_n=10):
    """Inspect label files: 5 columns -> bbox (detection), >5 columns -> polygon (segmentation)."""
    if not os.path.exists(labels_dir):
        return "unknown"
    txts = [f for f in os.listdir(labels_dir) if f.endswith(".txt")][:sample_n]
    for f in txts:
        with open(os.path.join(labels_dir, f)) as fh:
            for line in fh:
                parts = line.strip().split()
                if not parts:
                    continue
                if len(parts) == 5:
                    return "detection"
                elif len(parts) > 5:
                    return "segmentation"
    return "unknown"

# Try manual overrides first (failsafe if config cell wasn't executed)
DET_TOTAL_DIR = globals().get('DET_TOTAL_DIR_OVERRIDE', None)
SEG_TOTAL_DIR = globals().get('SEG_TOTAL_DIR_OVERRIDE', None)

if DET_TOTAL_DIR is None or SEG_TOTAL_DIR is None:
    candidates = find_candidate_dirs(EXTRACT_DIR)
    print(f"\nFound {len(candidates)} candidate image/label folder(s):")
    for c in candidates:
        images_path = next((os.path.join(c, d) for d in os.listdir(c) if d.lower() == "images"), None)
        labels_path = next((os.path.join(c, d) for d in os.listdir(c) if d.lower() == "labels"), None)

        n_images = len([f for f in os.listdir(images_path)
                         if os.path.splitext(f)[1].lower() in IMG_EXTS]) if images_path else 0
        label_type = detect_label_type(labels_path) if labels_path else "unknown"
        print(f"  {c}  ->  {n_images} images, label_type={label_type}")

        if label_type == "detection" and DET_TOTAL_DIR is None:
            DET_TOTAL_DIR = c
        if label_type == "segmentation" and SEG_TOTAL_DIR is None:
            SEG_TOTAL_DIR = c

if DET_TOTAL_DIR is None or SEG_TOTAL_DIR is None:
    # Print out directory contents to help user diagnose what was extracted
    print("\n" + "!" * 50)
    print("DEBUG DIAGNOSTIC: Listing files recursively in extract folder:")
    print("!" * 50)
    for root, dirs, files in os.walk(EXTRACT_DIR):
        depth = root.replace(EXTRACT_DIR, "").count(os.sep)
        if depth < 4:
            indent = "  " * depth
            print(f"{indent}[Dir] {os.path.basename(root) or 'raw_data'}/")
            for d in sorted(dirs):
                print(f"{indent}  ↳ [Subdir] {d}/")
            for f in sorted(files)[:15]:  # show up to 15 files
                print(f"{indent}  ↳ [File] {f}")
            if len(files) > 15:
                print(f"{indent}  ↳ ... and {len(files) - 15} more files")

    raise RuntimeError(
        "Could not auto-detect both DETECTION and SEGMENTATION folders.\n"
        "Please check the DEBUG DIAGNOSTIC output above to see what was actually extracted.\n"
        f"Candidates found: {candidates}"
    )

print(f"\nUsing DETECTION folder:   {DET_TOTAL_DIR}")
print(f"Using SEGMENTATION folder: {SEG_TOTAL_DIR}")


Found 2 candidate image/label folder(s):
  /content/carixray/raw_data/Total-Data/DETECTION/DETECTION  ->  4658 images, label_type=detection
  /content/carixray/raw_data/Total-Data/SEGMENTATION/SEGMENTATION  ->  4658 images, label_type=segmentation

Using DETECTION folder:   /content/carixray/raw_data/Total-Data/DETECTION/DETECTION
Using SEGMENTATION folder: /content/carixray/raw_data/Total-Data/SEGMENTATION/SEGMENTATION


In [6]:
#3. K-Fold split (same logic as repo's kfold/split_kfold.py, 80/20, seeded)
# ----------------------------------------------------------------------------
def atoi(text):
    return int(text) if text.isdigit() else text

def natural_keys(text):
    return [atoi(c) for c in re.split(r"(\d+)", text)]

def list_item(path):
    items = os.listdir(path)
    items.sort(key=natural_keys)
    return items

def make_folder(path):
    os.makedirs(path, exist_ok=True)

def split_kfold(images_dir, labels_dir, out_dir, n_folds=N_FOLDS, seed=SEED):
    make_folder(out_dir)
    item_list = [f for f in list_item(images_dir)
                  if os.path.splitext(f)[1].lower() in IMG_EXTS]

    rng = random.Random(seed)
    rng.shuffle(item_list)

    num_images = len(item_list)
    fold_size = num_images // n_folds
    remainder = num_images % n_folds

    start = 0
    for i in range(n_folds):
        fold_path = os.path.join(out_dir, f"fold_{i}")
        for split in ("train", "test"):
            for sub in ("images", "labels"):
                make_folder(os.path.join(fold_path, split, sub))

        end = start + fold_size + (1 if i < remainder else 0)
        test_images = set(item_list[start:end])

        for img in item_list:
            split = "test" if img in test_images else "train"
            label_name = os.path.splitext(img)[0] + ".txt"

            shutil.copy(os.path.join(images_dir, img),
                        os.path.join(fold_path, split, "images", img))

            src_label = os.path.join(labels_dir, label_name)
            if os.path.exists(src_label):
                shutil.copy(src_label,
                             os.path.join(fold_path, split, "labels", label_name))

        start = end
        print(f"  fold_{i}: {len(test_images)} test images, "
              f"{num_images - len(test_images)} train images")

print("\nSplitting DETECTION data into folds...")
split_kfold(os.path.join(DET_TOTAL_DIR, "images"),
            os.path.join(DET_TOTAL_DIR, "labels"),
            DET_KFOLD_DIR)

print("\nSplitting SEGMENTATION data into folds...")
split_kfold(os.path.join(SEG_TOTAL_DIR, "images"),
            os.path.join(SEG_TOTAL_DIR, "labels"),
            SEG_KFOLD_DIR)



Splitting DETECTION data into folds...
  fold_0: 932 test images, 3726 train images
  fold_1: 932 test images, 3726 train images
  fold_2: 932 test images, 3726 train images
  fold_3: 931 test images, 3727 train images
  fold_4: 931 test images, 3727 train images

Splitting SEGMENTATION data into folds...
  fold_0: 932 test images, 3726 train images
  fold_1: 932 test images, 3726 train images
  fold_2: 932 test images, 3726 train images
  fold_3: 931 test images, 3727 train images
  fold_4: 931 test images, 3727 train images


In [7]:
 #4. Write per-fold YOLO data.yaml files
# ----------------------------------------------------------------------------
def write_yolo_yaml(fold_dir):
    yaml_path = os.path.join(fold_dir, "data.yaml")
    data = {
        "path": fold_dir,
        "train": "train/images",
        "val": "test/images",
        "nc": len(CLASS_NAMES),
        "names": CLASS_NAMES,
    }
    with open(yaml_path, "w") as f:
        yaml.dump(data, f)
    return yaml_path


In [8]:
 #5. Train YOLOv8 — Detection (5 folds)
# ----------------------------------------------------------------------------
from ultralytics import YOLO

det_results = []
print("\n" + "=" * 60)
print("TRAINING YOLOv8 DETECTION — 5-FOLD CV")
print("=" * 60)

for i in range(N_FOLDS):
    fold_dir = os.path.join(DET_KFOLD_DIR, f"fold_{i}")
    yaml_path = write_yolo_yaml(fold_dir)

    print(f"\n--- Detection fold {i} ---")
    try:
        model = YOLO(DET_MODEL)
        model.train(data=yaml_path, epochs=EPOCHS_DET, imgsz=IMG_SIZE,
                     project=os.path.join(ROOT, "runs/det"), name=f"fold_{i}",
                     exist_ok=True, verbose=False)
        metrics = model.val(data=yaml_path, split="val", verbose=False)
        map50 = float(metrics.box.map50)
        map5095 = float(metrics.box.map)
    except Exception as e:
        print(f"  ERROR on fold {i}: {e}")
        map50, map5095 = float("nan"), float("nan")

    print(f"  fold {i}: mAP50={map50:.4f}  mAP50-95={map5095:.4f}")
    det_results.append({"Fold": i + 1, "YOLOv8_mAP50": map50, "YOLOv8_mAP50-95": map5095})

det_df = pd.DataFrame(det_results)
det_csv_path = os.path.join(ROOT, "det.csv")
det_df.to_csv(det_csv_path, index=False)
print(f"\nSaved detection results -> {det_csv_path}")
print(det_df)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

TRAINING YOLOv8 DETECTION — 5-FOLD CV

--- Detection fold 0 ---
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/carixray/DETECTION/K_folds/fold_0/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=t

In [ ]:
#6. Train YOLOv8 — Segmentation (5 folds)
# ----------------------------------------------------------------------------
seg_results = []
print("\n" + "=" * 60)
print("TRAINING YOLOv8 SEGMENTATION — 5-FOLD CV")
print("=" * 60)

for i in range(N_FOLDS):
    fold_dir = os.path.join(SEG_KFOLD_DIR, f"fold_{i}")
    yaml_path = write_yolo_yaml(fold_dir)

    print(f"\n--- Segmentation fold {i} ---")
    try:
        model = YOLO(SEG_MODEL)
        model.train(data=yaml_path, epochs=EPOCHS_SEG, imgsz=IMG_SIZE,
                     project=os.path.join(ROOT, "runs/seg"), name=f"fold_{i}",
                     exist_ok=True, verbose=False)
        metrics = model.val(data=yaml_path, split="val", verbose=False)
        seg_map50 = float(metrics.seg.map50)
        seg_map5095 = float(metrics.seg.map)
        box_map50 = float(metrics.box.map50)
    except Exception as e:
        print(f"  ERROR on fold {i}: {e}")
        seg_map50, seg_map5095, box_map50 = float("nan"), float("nan"), float("nan")

    print(f"  fold {i}: seg mAP50={seg_map50:.4f}  seg mAP50-95={seg_map5095:.4f}  "
          f"box mAP50={box_map50:.4f}")
    seg_results.append({
        "Fold": i + 1,
        "YOLOv8seg_mAP50": seg_map50,
        "YOLOv8seg_mAP50-95": seg_map5095,
        "YOLOv8seg_box_mAP50": box_map50,
    })

seg_df = pd.DataFrame(seg_results)
seg_csv_path = os.path.join(ROOT, "seg.csv")
seg_df.to_csv(seg_csv_path, index=False)
print(f"\nSaved segmentation results -> {seg_csv_path}")
print(seg_df)



TRAINING YOLOv8 SEGMENTATION — 5-FOLD CV

--- Segmentation fold 0 ---
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/carixray/SEGMENTATION/K_folds/fold_0/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fo

In [11]:
# Done & Final Summary Display
# ----------------------------------------------------------------------------
# Try to get results from global workspace (failsafe if only one task was run)
det_results_local = globals().get('det_results', [])
seg_results_local = globals().get('seg_results', [])

print("\n" + "=" * 70)
print("                           FINAL SUMMARY RESULTS")
print("=" * 70)

print("\n--- DETECTION FOLD ACCURACIES (mAP@0.5) ---")
if len(det_results_local) > 0:
    for res in det_results_local:
        print(f"  Fold {res['Fold']}: mAP50 = {res['YOLOv8_mAP50']:.4f} (mAP50-95 = {res['YOLOv8_mAP50-95']:.4f})")
    det_map50_vals = [r['YOLOv8_mAP50'] for r in det_results_local if not pd.isna(r['YOLOv8_mAP50'])]
    if det_map50_vals:
        import numpy as np
        print(f"  >> Average Detection mAP50: {np.mean(det_map50_vals):.4f} ± {np.std(det_map50_vals):.4f}")
else:
    print("  No detection results available.")

print("\n--- SEGMENTATION FOLD ACCURACIES (mAP@0.5) ---")
if len(seg_results_local) > 0:
    for res in seg_results_local:
        print(f"  Fold {res['Fold']}: Mask mAP50 = {res['YOLOv8seg_mAP50']:.4f} (Box mAP50 = {res['YOLOv8seg_box_mAP50']:.4f})")
    seg_map50_vals = [r['YOLOv8seg_mAP50'] for r in seg_results_local if not pd.isna(r['YOLOv8seg_mAP50'])]
    if seg_map50_vals:
        import numpy as np
        print(f"  >> Average Segmentation mAP50: {np.mean(seg_map50_vals):.4f} ± {np.std(seg_map50_vals):.4f}")
else:
    print("  No segmentation results available.")

print("\n" + "=" * 70)
print("PILOT COMPLETE")
print("=" * 70)
det_csv_path = os.path.join(ROOT, "det.csv")
seg_csv_path = os.path.join(ROOT, "seg.csv")
print(f"Detection results: {det_csv_path}")
print(f"Segmentation results: {seg_csv_path}")
print(f"Checkpoints under: {os.path.join(ROOT, 'runs')}")
print("\nTo extend this into the full benchmark, add more model columns to")
print("det.csv / seg.csv (one column per model, same Fold rows) and run the")
print("repo's significant-test/run.py for pairwise paired t-tests.")


                           FINAL SUMMARY RESULTS

--- DETECTION FOLD ACCURACIES (mAP@0.5) ---
  Fold 1: mAP50 = 0.8521 (mAP50-95 = 0.5137)
  Fold 2: mAP50 = 0.8384 (mAP50-95 = 0.4993)
  Fold 3: mAP50 = 0.8651 (mAP50-95 = 0.5141)
  Fold 4: mAP50 = 0.8660 (mAP50-95 = 0.5236)
  Fold 5: mAP50 = 0.8492 (mAP50-95 = 0.5090)
  >> Average Detection mAP50: 0.8542 ± 0.0103

--- SEGMENTATION FOLD ACCURACIES (mAP@0.5) ---
  No segmentation results available.

PILOT COMPLETE
Detection results: /content/carixray/det.csv
Segmentation results: /content/carixray/seg.csv
Checkpoints under: /content/carixray/runs

To extend this into the full benchmark, add more model columns to
det.csv / seg.csv (one column per model, same Fold rows) and run the
repo's significant-test/run.py for pairwise paired t-tests.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')